# SRQ-FLY D2.1 — nested lambda robustness
Run every cell in order on a Colab T4 GPU. Lambda selection uses only an inner split of D1 training. The selected exact FLY-4518 control is evaluated once on outer validation. `test.pt` must remain absent.

In [ ]:
# === Edit repository/Drive paths only. Do not edit grid, seed, split, or gates. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/srq-fly-d21-lambda-robustness'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/imagenetr_train_feature_cache_seed2025'
TRAIN_CACHE_DIR = '/content/imagenetr_train_feature_cache_seed2025'
D2_RESULT_PATH = f'{DRIVE_ROOT}/srq_fly_imagenetr_d2_state_match_seed2025/d2_results.json'
DRIVE_WTA_CACHE = f'{DRIVE_ROOT}/srq_fly_wta_h4518_seed2025'
WTA_CACHE_DIR = '/content/srq_fly_wta_h4518_seed2025'
OUTPUT_DIR = f'{DRIVE_ROOT}/srq_fly_imagenetr_d21_lambda_seed2025'
SEED = 2025
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
D2_RESULT_SHA256 = 'a7f08b4608e9f571da7698bab35bf64d967a18f01b9b2e4818ad1ee99a535263'
CONFIG_SHA256 = '3c5b54ffedacf5620c8cd9123acb187f5cbf958023b37aebecf9f00c45f73e96'

In [ ]:
# Runtime setup. chdir first to avoid Colab's deleted-working-directory failure.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo_path = Path(WORK_DIR)
if repo_path.exists(): shutil.rmtree(repo_path)
clone = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], text=True, capture_output=True)
print(clone.stdout, clone.stderr, sep='')
assert clone.returncode == 0, f'Clone failed ({clone.returncode}). Confirm the D2.1 branch was pushed.'
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/srq_fly_imagenetr_d21_lambda_robustness.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256, 'Locked config identity mismatch.'
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('locked seed:', SEED, '| config SHA-256:', CONFIG_SHA256)

In [ ]:
# Restore verified train/D2/WTA artifacts with bounded progress output.
def copy_with_progress(source, target, chunk_size=32*2**20):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and target.stat().st_size == source.stat().st_size:
        print('RESTORED', source.name, f'{source.stat().st_size/2**20:.1f} MiB', flush=True)
        return
    partial = target.with_suffix(target.suffix + '.partial')
    partial.unlink(missing_ok=True)
    copied, total, next_report = 0, source.stat().st_size, 10
    with source.open('rb') as reader, partial.open('wb') as writer:
        while True:
            block = reader.read(chunk_size)
            if not block: break
            writer.write(block); copied += len(block)
            percent = int(100*copied/max(total, 1))
            if percent >= next_report or copied == total:
                print(f'COPY {source.name} {percent:3d}% ({copied/2**20:.1f}/{total/2**20:.1f} MiB)', flush=True)
                next_report += 10
    assert copied == total
    partial.replace(target)
drive_train, local_train = Path(DRIVE_TRAIN_CACHE), Path(TRAIN_CACHE_DIR)
for name in ['metadata.json', 'train.pt']:
    source = drive_train/name
    assert source.is_file(), f'Missing required train cache file: {source}'
    copy_with_progress(source, local_train/name)
metadata = json.loads((local_train/'metadata.json').read_text())
assert metadata['dataset'] == 'ImageNet-R' and metadata['checkpoint_sha256'] == CHECKPOINT_SHA256
assert metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['test_features_materialized'] is False and not (local_train/'test.pt').exists()
d2_path = Path(D2_RESULT_PATH)
assert d2_path.is_file() and hashlib.sha256(d2_path.read_bytes()).hexdigest() == D2_RESULT_SHA256
drive_wta, local_wta = Path(DRIVE_WTA_CACHE), Path(WTA_CACHE_DIR)
for name in ['metadata.json', 'projection.pt', 'train_codes.pt']:
    source = drive_wta/name
    assert source.is_file(), f'Missing verified D2 WTA cache file: {source}'
    copy_with_progress(source, local_wta/name)
print('D2.1 preflight: PASS | D2 artifact locked | test.pt absent')

In [ ]:
# Correctness/leakage/resume gate: synthetic data only.
tests = ['tests/test_srq_fly_math.py', 'tests/test_srq_fly_d2_state_match.py', 'tests/test_srq_fly_d21_lambda_robustness.py']
subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], check=True)
print('SRQ-FLY D2.1 correctness gate: PASS')

In [ ]:
# Locked nested study. INNER lines select lambda; OUTER lines evaluate the locked winner once.
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
shutil.copy2(config_path, output_path/'locked_config.json')
command = [sys.executable, '-u', 'tools/srq_fly_d21_lambda_robustness.py', '--config', str(config_path), '--feature-cache-dir', TRAIN_CACHE_DIR, '--code-cache-dir', WTA_CACHE_DIR, '--d2-result', D2_RESULT_PATH, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting D2.1: 6 inner lambda candidates, then one locked outer evaluation.', flush=True)
print('Wait for INNER DONE lines and one OUTER DONE line.', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'Runner elapsed: {(time.time()-started)/60:.1f} minutes', flush=True)
assert completed.returncode == 0, 'D2.1 failed; send the complete traceback without editing config.'
assert (output_path/'d21_results.json').is_file() and not (local_train/'test.pt').exists()
print('SRQ-FLY D2.1 process: COMPLETE')

In [ ]:
# Summarize, download evidence, then STOP.
selection = json.loads((output_path/'lambda_selection.json').read_text())
for item in selection['candidates']:
    print(f"lambda={item['ridge_lambda']:g} inner_AA={item['validation_average_accuracy']:.6f} residual={item['maximum_solver_relative_residual']:.3e}")
result = json.loads((output_path/'d21_results.json').read_text())
control = result['tuned_state_matched_exact_fly']
reference = result['d2_reference']
print('selected lambda:', result['selection']['selected_lambda'])
print('tuned FLY-4518 outer AA/final:', control['validation_average_accuracy'], control['stage_accuracy'][-1])
print('SRQ outer AA/final:', reference['srq_validation_average_accuracy'], reference['srq_final_accuracy'])
print('comparison:', json.dumps(result['comparison'], indent=2))
print('decision:', result['status'])
print('gates:', json.dumps(result['gates'], indent=2))
archive = shutil.make_archive('/content/srq_fly_imagenetr_d21_lambda_robustness', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Send the ZIP for audit; do not evaluate ImageNet-R test.')